# Auditoria do de-para ICTV/NCBI versus split de fragmentos

Lê **o Parquet real** `main_taxonomy_hybrid_CLEANSED.parquet` produzido por `short/ICTV_improve_kraken_database.ipynb`, confere sua identidade de rótulos com `dataset_kraken/species_name_ids.csv` e cruza as classes com `per_species.csv` produzido pela auditoria de treino/teste. Estratifica duplicatas exatas por espécie, gênero, família e ordem.

A planilha ICTV e o fallback do NCBI aqui descrevem **táxons** (uma linha por espécie), não accessions de genomas de origem dos 15 milhões de fragmentos. Portanto esta auditoria identifica problemas do de-para e distribuição taxonômica da sobreposição, mas **não certifica independência por genoma e não toma a decisão final de retreinamento**. Não usa dados simulados nem chama a API do NCBI.

## Entradas e namespace M01

Configure `M01_CONFIG` com um YAML (modelo em `configs/reproduction_m01.yaml`) ou `M01_RUN_PLAN` com o plano M00 selecionado. O plano fixa entradas e destino; overrides conflitantes são recusados. Sem plano, cada execução reserva novo `run_id`. `DATA_ROOT`, `ORIGINAL_MAPPING_PATH`, `TAXONOMY_PATH` e os overrides descritos em `docs/m01_data_taxonomy_split.md` são aceitos sem plano. Saídas históricas permanecem somente leitura.

O modo padrão é empírico, inclusive se faltarem dados. Testes locais usam `REMOTE_SERVER=False SMOKE_TEST=1` com fixtures explícitas. Manifesto COMPLETE significa análise concluída, não integridade taxonômica nem independência genômica. Examine o relatório e `hierarchy_ready` antes de usar a hierarquia.


In [ ]:
import os
import sys
from pathlib import Path

current_dir = Path.cwd().resolve()
for r in [current_dir, current_dir.parent, current_dir.parent.parent, current_dir.parent.parent.parent]:
    if (r / "src" / "viral_cnn").exists():
        if str(r / "src") not in sys.path:
            sys.path.insert(0, str(r / "src"))
        PROJECT_ROOT = r
        break
else:
    raise FileNotFoundError("Execute dentro do repositório cnn_metagenome")

from viral_cnn.m01_workflow import run_m01_notebook


In [ ]:
manifest = run_m01_notebook("short/11", PROJECT_ROOT)
print("Run:", manifest["run_id"])
print("Saída:", manifest["output_dir"])
print("Status:", manifest["status"])


## Evidência e próximo passo

O manifesto registra caminhos, hashes ou identidade limitada explícita, configuração, versões e origem sintética/observada. Consulte `output_dir` abaixo; não procure a execução mais recente por data. A taxa de overlap usa o teste integral como denominador; piloto não estima prevalência. A ausência de duplicata exata não prova independência genômica. Estes resultados não são o benchmark C03.


In [ ]:
import json
print(json.dumps(manifest["metrics_summary"], indent=2, ensure_ascii=False))
